In [13]:
%pip install pycosat
import pycosat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pycosat: filename=pycosat-0.6.6-cp312-cp312-linux_x86_64.whl size=170300 sha256=25657dec85441b8339a10ffc5422fd3a8b73f41f84b8ad88b380e01160175f38
  Stored in directory: /root/.cache/pip/wheels/a2/34/2e/81095f4bfa4d06004e3bebc7e415733c40f0d0ab583100d3af
Successfully built pycosat


In [7]:
# exemplos nonograma

from typing import TypedDict

class Nonograma(TypedDict):
    linhas: list[list[int]]
    colunas: list[list[int]]

nonograma_simples = {
    "linhas": [
        [1, 1],
        [1, 1, 1],
        [1, 1],
        [1, 1],
        [1],
    ],
    "colunas": [
        [2],
        [1, 1],
        [1, 1],
        [1, 1],
        [2],
    ]
}

In [50]:
from itertools import combinations_with_replacement
from typing import List

class ResolvedorNonograma:
    def __init__(self, nonograma: Nonograma):
        self.m = len(nonograma["linhas"])
        self.n = len(nonograma["colunas"])
        self.restricoes_linhas = nonograma["linhas"]
        self.restricoes_colunas = nonograma["colunas"]

        self.cnf = []
        self.var = self.m * self.n + 1 # próxima variável auxiliar
        self.sol = None

    def x(self, i: int, j: int) -> int:
        return i * self.n + j + 1

    def var_aux(self) -> int:
        v = self.var
        self.var += 1
        return v

    def permutacoes(self, n: int, restricoes: List[int]) -> List[List[bool]]:
        if not restricoes:
            return [[False] * n]

        n_blocos = len(restricoes)
        folga = n - sum(restricoes) - (n_blocos - 1)

        if folga < 0:
            return None

        n_baldes = n_blocos + 1 # número de "baldes" nos quais pode-se distribuir a folga

        permutacoes = []

        # gera combinações de onde cada folga pode cair entre os blocos
        for p in combinations_with_replacement(range(n_baldes), folga):
            qtd_baldes = [p.count(i) for i in range(n_baldes)]

            linha = []
            for i in range(n_blocos):
                linha.extend([False] * qtd_baldes[i]) # folga antes do bloco atual
                linha.extend([True] * restricoes[i]) # bloco atual
                if i < n_blocos - 1:
                    linha.append(False) # espaço entre os blocos

            # folga após o último bloco
            linha.extend([False] * qtd_baldes[-1])

            permutacoes.append(linha)
        return permutacoes

    def gera_cnf_eixo(self, restricoes: List[int], vars: List[int]):
        # gera cnf para linha ou coluna

        n = len(vars)

        if restricoes == []: # linha / coluna sem restrições
            for v in vars:
                self.cnf.append([-v])
            return True

        permutacoes = self.permutacoes(n, restricoes)
        if permutacoes == None:
            return False

        # variáveis auxiliares para a transformação de tseitin
        vars_aux = [self.var_aux() for _ in range(len(permutacoes))]
        self.cnf.append(vars_aux) # raíz da árvore, uma das permutações deve ser verdadeira

        for i, p in enumerate(permutacoes):
            w = vars_aux[i]

            # p (permutação) -> w (cláusula auxiliar)
            # disjunção da negação das variáveis de p v w
            p_entao_w = [w]

            for j in range(n):
                if p[j]:
                    p_entao_w.append(-vars[j])

                    # w (cláusula auxiliar) -> p (permutação)
                    self.cnf.append([-w, vars[j]])
                else:
                    p_entao_w.append(vars[j])

                    # w (cláusula auxiliar) -> p (permutação)
                    self.cnf.append([-w, -vars[j]])

            self.cnf.append(p_entao_w)
        return True

    def gera_cnf(self):
        for i in range(self.m):
            vars_linha = [self.x(i, j) for j in range(self.n)]
            if not self.gera_cnf_eixo(self.restricoes_linhas[i], vars_linha):
                self.sol = "UNSAT"
                self.reset()
                return False

        for j in range(self.n):
            vars_coluna = [self.x(i, j) for i in range(self.m)]
            if not self.gera_cnf_eixo(self.restricoes_colunas[j], vars_coluna):
                self.sol = "UNSAT"
                self.reset()
                return False

        return True

    def resolve(self):
        if not self.gera_cnf():
            self.reset()
            self.cnf = []
            return

        output = pycosat.solve(self.cnf)
        if output == "UNSAT":
            self.sol = "UNSAT"
            return

        output_truncado = output[:self.m * self.n]
        output_bool = [True if var > 0 else False for var in output_truncado]
        self.sol = [output_bool[i : i + self.n] for i in range(0, len(output_bool), self.n)]

    def resultado(self):
        if not self.sol:
            print("chame o resolvedor")
            return

        if isinstance(self.sol, str):
            print(self.sol)
            return

        max_linhas = max((len(r) for r in self.restricoes_linhas), default=0)
        max_colunas = max((len(c) for c in self.restricoes_colunas), default=0)

        # restrições colunas
        for i in range(max_colunas):
            linha_str = " " * (max_linhas * 2) # recuo

            for j in range(self.n):
                restricoes = self.restricoes_colunas[j]
                offset = max_colunas - len(restricoes)

                if i < max_colunas - len(restricoes):
                    linha_str += "  "
                else:
                    linha_str += str(restricoes[i - offset]) + " "

            print(linha_str.rstrip())

        # restrições linhas + grade
        for i, row in enumerate(self.sol):
            restricoes = self.restricoes_linhas[i]

            restricoes_str = [str(x) for x in restricoes]
            padding = [" "] * (max_linhas - len(restricoes))
            linha_str = " ".join(padding + restricoes_str)

            linha_str = linha_str + " " + " ".join("■" if var else "□" for var in row)

            print(linha_str)

    def reset(self):
        self.cnf = []
        self.var = self.m * self.n + 1
        self.sol = None

In [51]:
resolvedor_simples = ResolvedorNonograma(nonograma_simples)

resolvedor_simples.resolve()
resolvedor_simples.resultado()

        1 1 1
      2 1 1 1 2
  1 1 □ ■ □ ■ □
1 1 1 ■ □ ■ □ ■
  1 1 ■ □ □ □ ■
  1 1 □ ■ □ ■ □
    1 □ □ ■ □ □
